# 02 — Crystal Lab

**No API key required.** Analyse crystal symmetry with spglib and compute grain sizes from XRD peak broadening.

Capabilities:
- Space group determination (230 space groups)
- Crystal system classification
- Scherrer grain-size estimation from XRD FWHM

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from chemvision.physics.symmetry import CrystalSymmetryAnalyzer
from chemvision.physics.scherrer import ScherrerAnalyzer

sym = CrystalSymmetryAnalyzer()
sch = ScherrerAnalyzer()
print('Physics modules ready')

## 2.1 — Analyse crystal symmetry

In [ ]:
crystals = {
    'FCC Cu': dict(a=3.615, b=3.615, c=3.615, alpha=90, beta=90, gamma=90,
                   species=[29,29,29,29],
                   fractional_positions=[[0,0,0],[.5,.5,0],[.5,0,.5],[0,.5,.5]]),
    'BCC Fe': dict(a=2.87, b=2.87, c=2.87, alpha=90, beta=90, gamma=90,
                   species=[26,26],
                   fractional_positions=[[0,0,0],[.5,.5,.5]]),
    'Rutile TiO2': dict(a=4.594, b=4.594, c=2.959, alpha=90, beta=90, gamma=90,
                        species=[22,22,8,8,8,8],
                        fractional_positions=[[0,0,0],[.5,.5,.5],[.305,.305,0],[.695,.695,0],[.805,.195,.5],[.195,.805,.5]]),
    'Diamond Si': dict(a=5.431, b=5.431, c=5.431, alpha=90, beta=90, gamma=90,
                       species=[14]*8,
                       fractional_positions=[[0,0,0],[.5,.5,0],[.5,0,.5],[0,.5,.5],[.25,.25,.25],[.75,.75,.25],[.75,.25,.75],[.25,.75,.75]]),
}

import pandas as pd
rows = []
for name, params in crystals.items():
    r = sym.from_lattice_params(**params)
    rows.append({
        'Crystal': name,
        'Space group': f'{r.space_group_symbol} (#{r.space_group_number})',
        'System': r.crystal_system.capitalize(),
        'Point group': r.point_group,
        'Valid': r.is_valid,
    })

pd.DataFrame(rows)

## 2.2 — Scherrer grain-size from XRD peaks

In [ ]:
# TiO2 anatase XRD peaks: (2theta, FWHM) in degrees
anatase_peaks = [(25.3, 0.35), (48.0, 0.42), (53.8, 0.45), (55.1, 0.50), (62.7, 0.55)]

results = sch.analyze_peaks(anatase_peaks)
mean_size = sch.mean_grain_size_nm(anatase_peaks)

print(f'Mean crystallite size: {mean_size:.1f} nm\n')

rows = []
for r in results:
    rows.append({
        '2theta (deg)': r.two_theta_deg,
        'FWHM (deg)': r.fwhm_deg,
        'Grain size (nm)': f'{r.grain_size_nm:.1f}' if r.valid else '—',
    })
pd.DataFrame(rows)

In [ ]:
# Visualise grain size per peak
valid = [r for r in results if r.valid]
x = [f'{r.two_theta_deg:.1f}' for r in valid]
y = [r.grain_size_nm for r in valid]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(x, y, color='steelblue', edgecolor='navy')
ax.axhline(mean_size, color='red', linestyle='--', label=f'Mean = {mean_size:.1f} nm')
ax.set_xlabel('2theta (deg)')
ax.set_ylabel('Grain size (nm)')
ax.set_title('Scherrer grain size — TiO2 anatase')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, y):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 2.3 — Compare anatase vs rutile grain sizes

In [ ]:
rutile_peaks = [(27.4, 0.30), (36.1, 0.38), (41.2, 0.40), (54.3, 0.48)]

anatase_mean = sch.mean_grain_size_nm(anatase_peaks)
rutile_mean = sch.mean_grain_size_nm(rutile_peaks)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Anatase', 'Rutile'], [anatase_mean, rutile_mean],
       color=['#4e79a7', '#e15759'], edgecolor='black')
ax.set_ylabel('Mean grain size (nm)')
ax.set_title('TiO2 phase comparison')
for i, v in enumerate([anatase_mean, rutile_mean]):
    ax.text(i, v + 0.5, f'{v:.1f} nm', ha='center', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()